# 3. Smoke Test

This notebook runs matched 50-step baseline and AttnRes smoke tests and compares the resulting summaries.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-mini.git'
REPO_NAME = 'AttnResGPT-mini'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

candidates = [Path(f'/content/{REPO_NAME}'), Path(f'/content/drive/MyDrive/{REPO_NAME}'), Path.cwd()]
repo_root = next((p for p in candidates if (p / 'requirements.txt').exists() and (p / 'src').exists()), None)

if repo_root is None:
    target = Path(f'/content/{REPO_NAME}')
    print(f'Cloning {REPO_URL} into {target} ...')
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    repo_root = target
else:
    print(f'Using existing repo at {repo_root}')

%cd {repo_root}
!pip -q install -r requirements.txt

In [ ]:
import torch
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name:', torch.cuda.get_device_name(0))

In [ ]:
!python -m src.train --config configs/pilot_t4.yaml --overrides experiment.name=nb3_smoke_baseline model.architecture=baseline training.max_steps=50 training.eval_interval=25 training.checkpoint_interval=50
!python -m src.train --config configs/pilot_t4.yaml --overrides experiment.name=nb3_smoke_attnres model.architecture=attnres model.attnres.enabled=true training.max_steps=50 training.eval_interval=25 training.checkpoint_interval=50

In [ ]:
from pathlib import Path

baseline_run = sorted(Path('runs').glob('nb3_smoke_baseline_*'))[-1]
attnres_run = sorted(Path('runs').glob('nb3_smoke_attnres_*'))[-1]
!python scripts/compare_runs.py --baseline-run {baseline_run} --attnres-run {attnres_run}